# Web Scraping Climático — Pronóstico 5 Días
**Fuente:** [OpenWeatherMap API](https://openweathermap.org/) — `/data/2.5/forecast`  
**Tecnología:** PySpark + Requests  
**Cobertura:** Principales ciudades de Colombia

---
## ¿Cómo funciona la API?

OpenWeatherMap expone el pronóstico de 5 días en bloques de **3 horas** (40 registros por ciudad) mediante un endpoint REST:

```
GET https://api.openweathermap.org/data/2.5/forecast
    ?q=Bogota,CO
    &appid=TU_API_KEY
    &units=metric      ← temperaturas en °C
    &lang=es           ← descripciones en español
    &cnt=40            ← máximo 40 bloques (5 días × 8 bloques/día)
```

No requiere scraping HTML — la respuesta ya viene estructurada en JSON.


## 1. Instalación de dependencias

In [0]:
# !pip install pyspark requests matplotlib pandas

## 2. Imports

In [0]:
import requests
import json
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, LongType
)

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np

print("Imports listos")

## 3. Inicializar SparkSession

In [0]:
spark = (
    SparkSession.builder
    .appName("WebScraper_Clima_Colombia")
    .config("spark.sql.repl.eagerEval.enabled", "true")
    .getOrCreate()
)
print("SparkSession iniciada")

## 4. Configuración

In [0]:
API_KEY = "0604c8e09ac3049afa0115aad9e33c3e"   # key "procesamiento"
BASE_URL = "https://api.openweathermap.org/data/2.5/forecast"

# Principales ciudades de Colombia con su código de país
CIUDADES = [
    "Bogota,CO",
    "Medellin,CO",
    "Cali,CO",
    "Barranquilla,CO",
    "Cartagena,CO",
    "Bucaramanga,CO",
    "Manizales,CO",
    "Pereira,CO",
    "Cucuta,CO",
    "Ibague,CO",
    "Pasto,CO",
    "Neiva,CO",
]

print(f" Ciudades a consultar: {len(CIUDADES)}")
for c in CIUDADES:
    print(f"   • {c}")

## 5. Web Scraping — Extracción de datos

Cada llamada a la API devuelve **40 registros** (pronóstico cada 3h durante 5 días) con la siguiente estructura JSON:

```json
{
  "city": { "name": "Bogota", "country": "CO", "population": 7674366 },
  "list": [
    {
      "dt": 1712592000,              ← timestamp Unix
      "dt_txt": "2024-04-08 12:00:00",
      "main": {
        "temp": 14.5,               ← temperatura actual °C
        "temp_min": 13.2,
        "temp_max": 15.8,
        "feels_like": 13.9,
        "humidity": 82,             ← humedad %
        "pressure": 1013
      },
      "weather": [{ "description": "lluvia ligera", "icon": "10d" }],
      "wind": { "speed": 3.2, "deg": 180 },
      "pop": 0.72,                  ← probabilidad de precipitación (0-1)
      "clouds": { "all": 90 }       ← cobertura nubosa %
    }
  ]
}
```


In [0]:
def fetch_forecast(ciudad: str, api_key: str) -> list[dict]:
    """
    Obtiene el pronóstico de 5 días para una ciudad.
    Devuelve lista de registros planos (un dict por bloque de 3h).
    """
    params = {
        "q"     : ciudad,
        "appid" : api_key,
        "units" : "metric",   # °C
        "lang"  : "es",
        "cnt"   : 40,         # 5 días × 8 bloques
    }
    r = requests.get(BASE_URL, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()

    ciudad_nombre = data["city"]["name"]
    poblacion     = data["city"].get("population", 0)
    registros     = []

    for item in data["list"]:
        registros.append({
            "ciudad"          : ciudad_nombre,
            "poblacion"       : poblacion,
            "timestamp"       : item["dt"],
            "fecha_hora"      : item["dt_txt"],
            "temp"            : item["main"]["temp"],
            "temp_min"        : item["main"]["temp_min"],
            "temp_max"        : item["main"]["temp_max"],
            "sensacion"       : item["main"]["feels_like"],
            "humedad"         : item["main"]["humidity"],
            "presion"         : item["main"]["pressure"],
            "descripcion"     : item["weather"][0]["description"],
            "viento_vel"      : item["wind"]["speed"],
            "viento_dir"      : item["wind"].get("deg", 0),
            "prob_lluvia"     : round(item.get("pop", 0) * 100, 1),
            "nubosidad"       : item["clouds"]["all"],
        })
    return registros


# ── Ejecutar scraping para todas las ciudades ──
all_records = []
print("Consultando OpenWeatherMap...\n")

for ciudad in CIUDADES:
    try:
        records = fetch_forecast(ciudad, API_KEY)
        all_records.extend(records)
        print(f"{ciudad:<20} → {len(records)} registros")
    except Exception as e:
        print(f"{ciudad:<20} → Error: {e}")

print(f"\nTotal registros obtenidos: {len(all_records):,}")

## 6. Crear DataFrame en PySpark

In [0]:
schema = StructType([
    StructField("ciudad",      StringType(),  True),
    StructField("poblacion",   LongType(),    True),
    StructField("timestamp",   LongType(),    True),
    StructField("fecha_hora",  StringType(),  True),
    StructField("temp",        DoubleType(),  True),
    StructField("temp_min",    DoubleType(),  True),
    StructField("temp_max",    DoubleType(),  True),
    StructField("sensacion",   DoubleType(),  True),
    StructField("humedad",     IntegerType(), True),
    StructField("presion",     IntegerType(), True),
    StructField("descripcion", StringType(),  True),
    StructField("viento_vel",  DoubleType(),  True),
    StructField("viento_dir",  IntegerType(), True),
    StructField("prob_lluvia", DoubleType(),  True),
    StructField("nubosidad",   IntegerType(), True),
])

df = spark.createDataFrame(all_records, schema=schema)

# Extraer fecha y hora por separado
df = df.withColumn("fecha", F.to_date(F.col("fecha_hora")))
df = df.withColumn("hora",  F.hour(F.to_timestamp(F.col("fecha_hora"))))

df.createOrReplaceTempView("clima")

print("Schema:")
df.printSchema()
print(f"\n   Filas     : {df.count():,}")
print(f"   Columnas  : {len(df.columns)}")
print(f"   Ciudades  : {df.select('ciudad').distinct().count()}")

## 7. Análisis con Spark SQL

In [0]:
# ── Análisis 1: Resumen por ciudad ──
print("=" * 60)
print("  Resumen climático por ciudad (promedio 5 días)")
print("=" * 60)
df_resumen = spark.sql("""
    SELECT ciudad,
           ROUND(AVG(temp), 1)        AS temp_prom,
           ROUND(MIN(temp_min), 1)    AS temp_minima,
           ROUND(MAX(temp_max), 1)    AS temp_maxima,
           ROUND(AVG(humedad), 1)     AS humedad_prom,
           ROUND(AVG(prob_lluvia), 1) AS prob_lluvia_prom,
           ROUND(AVG(viento_vel), 1)  AS viento_prom_ms
    FROM   clima
    GROUP  BY ciudad
    ORDER  BY temp_prom DESC
""")
df_resumen.show(truncate=False)

In [0]:
# ── Análisis 2: Pronóstico día a día por ciudad ──
print("=" * 60)
print("  Temperatura promedio por día — Bogotá")
print("=" * 60)
df_dias = spark.sql("""
    SELECT ciudad,
           fecha,
           ROUND(AVG(temp), 1)        AS temp_prom,
           ROUND(MIN(temp_min), 1)    AS temp_min,
           ROUND(MAX(temp_max), 1)    AS temp_max,
           ROUND(AVG(humedad), 0)     AS humedad,
           ROUND(AVG(prob_lluvia), 0) AS prob_lluvia
    FROM   clima
    WHERE  ciudad = 'Bogota'
    GROUP  BY ciudad, fecha
    ORDER  BY fecha
""")
df_dias.show(truncate=False)

In [0]:
# ── Análisis 3: Ciudades más lluviosas ──
print("=" * 60)
print("  Probabilidad de lluvia promedio por ciudad")
print("=" * 60)
df_lluvia = spark.sql("""
    SELECT ciudad,
           ROUND(AVG(prob_lluvia), 1)  AS prob_lluvia_prom,
           ROUND(AVG(nubosidad), 1)    AS nubosidad_prom,
           COUNT(CASE WHEN prob_lluvia > 50 THEN 1 END) AS bloques_lluvia
    FROM   clima
    GROUP  BY ciudad
    ORDER  BY prob_lluvia_prom DESC
""")
df_lluvia.show(truncate=False)

In [0]:
# ── Análisis 4: Variación de temperatura por hora del día ──
print("=" * 60)
print("  Temperatura promedio por hora del día (todas las ciudades)")
print("=" * 60)
df_horas = spark.sql("""
    SELECT hora,
           ROUND(AVG(temp), 1)     AS temp_prom,
           ROUND(AVG(humedad), 1)  AS humedad_prom
    FROM   clima
    GROUP  BY hora
    ORDER  BY hora
""")
df_horas.show(truncate=False)

## 8. Visualizaciones

In [0]:
# Paleta de colores
AZUL    = "#1a73e8"
ROJO    = "#ea4335"
VERDE   = "#34a853"
NARANJA = "#fa7b17"
MORADO  = "#9334e6"

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.suptitle(
    "Pronóstico Climático 5 Días — Principales Ciudades de Colombia\n"
    "Fuente: OpenWeatherMap API | Datos en tiempo real",
    fontsize=14, fontweight="bold", y=0.98
)

pd_resumen = df_resumen.toPandas()
pd_dias    = df_dias.toPandas()
pd_lluvia  = df_lluvia.toPandas()
pd_horas   = df_horas.toPandas()

# ── Gráfico 1: Temperatura promedio por ciudad (con rango min-max) ──
ax1 = axes[0, 0]
y    = range(len(pd_resumen))
ax1.barh(pd_resumen["ciudad"], pd_resumen["temp_prom"],
         color=NARANJA, edgecolor="white", alpha=0.85, label="Prom")
ax1.errorbar(
    pd_resumen["temp_prom"], list(y),
    xerr=[
        pd_resumen["temp_prom"] - pd_resumen["temp_minima"],
        pd_resumen["temp_maxima"] - pd_resumen["temp_prom"]
    ],
    fmt="none", color="#333", capsize=4, linewidth=1.5
)
ax1.set_xlabel("Temperatura (°C)")
ax1.set_title("Temperatura Promedio por Ciudad\n(barras de error = rango min/max)")
ax1.invert_yaxis()
for bar, val in zip(ax1.patches, pd_resumen["temp_prom"]):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f"{val}°C", va="center", fontsize=8, fontweight="bold")

# ── Gráfico 2: Probabilidad de lluvia por ciudad ──
ax2 = axes[0, 1]
colors_bar = [AZUL if p < 40 else NARANJA if p < 60 else ROJO
              for p in pd_lluvia["prob_lluvia_prom"]]
bars = ax2.bar(pd_lluvia["ciudad"], pd_lluvia["prob_lluvia_prom"],
               color=colors_bar, edgecolor="white")
ax2.axhline(50, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Umbral 50%")
ax2.set_ylabel("Probabilidad de Lluvia (%)")
ax2.set_title("Probabilidad Promedio de Lluvia por Ciudad")
ax2.set_ylim(0, 100)
ax2.tick_params(axis="x", rotation=45)
ax2.legend()
for bar, val in zip(bars, pd_lluvia["prob_lluvia_prom"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{val}%", ha="center", fontsize=8, fontweight="bold")

# ── Gráfico 3: Pronóstico día a día Bogotá (temp + prob lluvia) ──
ax3 = axes[1, 0]
ax3b = ax3.twinx()

pd_dias["fecha"] = pd.to_datetime(pd_dias["fecha"])
ax3.fill_between(pd_dias["fecha"], pd_dias["temp_min"], pd_dias["temp_max"],
                 alpha=0.2, color=NARANJA, label="Rango temp")
ax3.plot(pd_dias["fecha"], pd_dias["temp_prom"],
         marker="o", color=NARANJA, linewidth=2.5,
         markersize=7, label="Temp prom (°C)")
ax3b.bar(pd_dias["fecha"], pd_dias["prob_lluvia"],
         alpha=0.35, color=AZUL, width=0.6, label="Prob. lluvia (%)")

ax3.set_ylabel("Temperatura (°C)", color=NARANJA)
ax3b.set_ylabel("Prob. Lluvia (%)", color=AZUL)
ax3.set_title("Pronóstico Día a Día — Bogotá")
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax3.tick_params(axis="x", rotation=30)

lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

# ── Gráfico 4: Temperatura y humedad por hora del día ──
ax4 = axes[1, 1]
ax4b = ax4.twinx()

horas = pd_horas["hora"]
ax4.plot(horas, pd_horas["temp_prom"],
         marker="o", color=NARANJA, linewidth=2.5,
         markersize=6, label="Temperatura (°C)")
ax4b.plot(horas, pd_horas["humedad_prom"],
          marker="s", color=AZUL, linewidth=2,
          markersize=6, linestyle="--", label="Humedad (%)")

ax4.set_xlabel("Hora del Día")
ax4.set_ylabel("Temperatura (°C)", color=NARANJA)
ax4b.set_ylabel("Humedad (%)", color=AZUL)
ax4.set_title("Variación Temperatura y Humedad por Hora\n(promedio todas las ciudades)")
ax4.set_xticks(range(0, 24, 3))
ax4.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 3)], rotation=30)
ax4.grid(axis="x", alpha=0.3)

lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("pronostico_colombia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado: pronostico_colombia.png")

## 9. Cerrar SparkSession

In [0]:
spark.stop()
print("SparkSession cerrada. Proceso finalizado.")